In [1]:
!pip install lifelines

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 13.2 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=c3dc2727684b76e74d299e8d040d1404e7f6bebdf6047f71ef653a0e2642d927
  Stored in directory: /root/.cache/pip/wheels/50/37/21/0a719b9d89c635e89ff24bd93b862882ad675279552013b2fb
Successfully built autograd-gamma


In [2]:
import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter
from tqdm.notebook import tqdm  # progress bar for bootstrap

# Data loading

In [3]:
# Real dataset
df_real = pd.read_parquet("/content/final_dataset.parquet")
df_real = df_real.convert_dtypes()
df_real = df_real.drop(columns="Patient_ID")
df_real.head()

,Sex,Age,BMI,Smoking_Status,ASA,T2DM,HTN,Dl,IHD,COPD,...,Pneumonia,Paralytic_ileus,Kidney_failure,Reintervention,Readmission,LOS_days,Survival_days,Recurrence_free_days,Death,Recurrence
0,0,84,30.12,1,3,0,1,0,0,0,...,0,0,0,1,0,27,1322,1322,1,0
1,1,85,34.72,0,3,0,1,1,0,0,...,0,1,0,0,0,11,235,123,1,0
2,1,85,28.96,0,2,0,0,0,0,0,...,0,0,0,0,0,4,2291,2291,0,0
3,1,84,28.3,0,3,0,1,1,1,0,...,0,0,0,0,0,16,1058,1058,0,0
4,1,64,38.37,0,2,1,0,1,0,0,...,0,0,0,1,1,19,2111,1385,0,1


In [4]:
# Gaussian Copula dataset
df_gc = pd.read_parquet("/content/copula_synth_best.parquet")
df_gc = df_gc.convert_dtypes()
df_gc = df_gc.drop(columns="Patient_ID")
df_gc.head()

,Sex,Age,BMI,Smoking_Status,ASA,T2DM,HTN,Dl,IHD,COPD,...,Pneumonia,Paralytic_ileus,Kidney_failure,Reintervention,Readmission,LOS_days,Survival_days,Recurrence_free_days,Death,Recurrence
0,1,61,27.1,0,2,1,1,1,0,0,...,0,1,0,0,0,11,2261,2516,0,1
1,0,84,27.11,0,2,0,1,1,0,0,...,0,0,0,0,0,8,855,448,1,0
2,0,86,24.89,1,2,0,1,1,1,0,...,0,1,0,0,0,16,24,15,1,1
3,1,74,30.86,0,3,0,0,1,0,0,...,0,1,0,0,0,1,3256,3141,0,0
4,0,71,31.44,0,2,1,1,0,0,0,...,0,0,0,0,0,5,3362,3362,0,0


In [5]:
# Variational Autoencoder dataset
df_vae = pd.read_parquet("/content/tvae_result_best.parquet")
df_vae = df_vae.convert_dtypes()
df_vae = df_vae.drop(columns="Patient_ID")
df_vae.head()

,Sex,Age,BMI,Smoking_Status,ASA,T2DM,HTN,Dl,IHD,COPD,...,Pneumonia,Paralytic_ileus,Kidney_failure,Reintervention,Readmission,LOS_days,Survival_days,Recurrence_free_days,Death,Recurrence
0,0,51,26.58,2,2,1,1,0,0,0,...,0,0,0,0,0,14,2676,2519,0,0
1,1,64,23.64,0,2,1,0,1,0,0,...,0,0,0,0,0,8,3267,2601,0,0
2,0,76,26.6,1,2,0,1,0,0,0,...,0,0,0,0,0,16,1129,831,1,1
3,1,81,23.72,1,2,0,0,0,0,0,...,0,0,0,0,0,6,2975,2423,0,0
4,1,76,23.2,0,2,1,1,0,0,0,...,0,0,0,0,0,7,1957,1852,1,0


In [6]:
# Generative Adversarial Network dataset
df_gan = pd.read_parquet("/content/synthetic_CTGAN.parquet")
df_gan = df_gan.convert_dtypes()
df_gan = df_gan.drop(columns="Patient_ID")
df_gan.head()

,Sex,Age,BMI,Smoking_Status,ASA,T2DM,HTN,Dl,IHD,COPD,...,Pneumonia,Paralytic_ileus,Kidney_failure,Reintervention,Readmission,LOS_days,Survival_days,Recurrence_free_days,Death,Recurrence
0,1,43,24.05,1,4,1,0,0,0,0,...,0,0,0,0,0,3,2405,2260,0,0
1,0,77,34.68,2,2,1,1,1,1,0,...,0,1,0,1,0,6,131,179,1,1
2,0,79,19.59,0,2,0,1,0,1,0,...,0,1,0,0,0,5,526,1888,0,1
3,1,88,24.53,1,2,0,1,0,0,0,...,0,0,0,0,0,6,251,197,0,0
4,1,71,30.36,0,<NA>,0,1,0,0,0,...,0,0,0,0,0,0,1803,265,0,1


In [7]:
# CopulaGAN dataset
df_gcgan = pd.read_parquet("/content/synthetic_final_CopulaGAN_seed42.parquet")
df_gcgan = df_gcgan.convert_dtypes()
df_gcgan = df_gcgan.drop(columns="Patient_ID")
df_gcgan.head()

,Sex,Age,BMI,Smoking_Status,ASA,T2DM,HTN,Dl,IHD,COPD,...,Pneumonia,Paralytic_ileus,Kidney_failure,Reintervention,Readmission,LOS_days,Survival_days,Recurrence_free_days,Death,Recurrence
0,0,61,29.75,1,2,1,1,1,0,1,...,0,0,0,0,0,9,2841,2389,0,0
1,0,69,25.26,0,3,1,1,1,0,0,...,0,0,0,0,1,37,3264,3122,0,0
2,1,58,24.7,2,2,0,0,0,0,1,...,0,0,0,0,0,35,434,83,1,0
3,1,51,21.17,1,3,0,0,0,0,0,...,0,0,0,0,0,5,2112,2572,1,1
4,0,66,25.15,1,3,0,0,0,0,0,...,0,0,0,0,0,5,2200,3252,0,0


# Preparation

In [8]:
time_col  = 'Survival_days'
event_col = 'Death'
timepoints = list(range(0, 3501, 500))

In [9]:
def fit_km(df, label):
    kmf = KaplanMeierFitter(label=label)
    kmf.fit(df[time_col], event_observed=df[event_col])
    return kmf

In [10]:
def bootstrap_km_median(duration, event_observed, n_bootstrap: int = 2000, random_state: int = 42):
    """
    Bootstrap 95% CI for the KM median survival time.

    Returns a dict with:
      point_estimate  – KM median from the full sample
      ci_lower        – 2·5th percentile of bootstrap distribution
      ci_upper        – 97·5th percentile of bootstrap distribution
      nr_fraction     – proportion of resamples where median was not reached
      n_estimable     – number of resamples that yielded a finite median

    Interpretation of nr_fraction
    ──────────────────────────────
      < 10 %   → median is stable; bootstrap CI is reliable.
      10–40 %  → borderline; CI is optimistic (computed on a selected subset).
               Consider reporting a supplementary metric (e.g. RMST).
      > 40 %   → median is unreliable; prefer RMST or survival at fixed time.
    """
    rng           = np.random.default_rng(random_state)
    duration      = np.asarray(duration, dtype=float)
    event_observed = np.asarray(event_observed, dtype=float)
    n             = len(duration)

    # Point estimate on the full sample
    kmf = KaplanMeierFitter()
    kmf.fit(duration, event_observed)
    point_estimate = kmf.median_survival_time_

    medians   = []
    nr_count  = 0

    for _ in range(n_bootstrap):
        idx    = rng.integers(0, n, size=n)
        kmf_b  = KaplanMeierFitter()
        kmf_b.fit(duration[idx], event_observed[idx])
        m = kmf_b.median_survival_time_
        if np.isinf(m) or np.isnan(m):
            nr_count += 1
        else:
            medians.append(m)

    medians     = np.array(medians)
    nr_fraction = nr_count / n_bootstrap
    n_estimable = len(medians)

    if n_estimable == 0:
        return dict(
            point_estimate = point_estimate,
            ci_lower       = np.nan,
            ci_upper       = np.nan,
            nr_fraction    = nr_fraction,
            n_estimable    = 0,
        )

    return dict(
        point_estimate = point_estimate,
        ci_lower       = float(np.percentile(medians, 2.5)),
        ci_upper       = float(np.percentile(medians, 97.5)),
        nr_fraction    = nr_fraction,
        n_estimable    = n_estimable,
    )


In [11]:
def get_numbers_at_risk(kmf, timepoints):
    return {
        t: int(kmf.event_table['at_risk']
                 .loc[kmf.event_table.index <= t]
                 .iloc[-1])
        if not kmf.event_table.loc[kmf.event_table.index <= t].empty else np.nan
        for t in timepoints
    }

In [12]:
def prepare_survival(df, time_col, event_col):
    tmp = df[[time_col, event_col]].copy()
    for col in (time_col, event_col):
        tmp[col] = pd.to_numeric(tmp[col], errors='coerce')
    tmp = tmp.dropna()
    tmp = tmp[tmp[time_col] >= 0]
    return tmp, [{'name': 'survival', 'time_col': time_col, 'event_col': event_col}]

# Risk table and median

In [14]:
def compute_km_statistics(datasets: dict, timepoints=timepoints, n_bootstrap: int = 2000, random_state: int = 42):
    """
    For each dataset, computes:
      – KM median with bootstrap 95% CI
      – NR fraction (diagnostic for median reliability)
      – numbers at risk at each timepoint

    Returns
    -------
    median_df  : DataFrame  (one row per dataset)
    risk_df    : DataFrame  (timepoints × datasets)
    """
    median_rows = []
    risk_rows   = {}

    for label, df in tqdm(datasets.items(), desc='Bootstrap KM'):
        # KM fit for numbers at risk
        kmf = fit_km(df, label)
        risk_rows[label] = get_numbers_at_risk(kmf, timepoints)

        # Bootstrap median CI
        boot = bootstrap_km_median(
            duration        = df[time_col],
            event_observed  = df[event_col],
            n_bootstrap     = n_bootstrap,
            random_state    = random_state,
        )

        def _fmt(v):
            return round(v, 1) if (v is not None and not np.isnan(v)) else 'NR'

        nr_pct = f"{boot['nr_fraction']:.1%}"
        if boot['nr_fraction'] > 0.40:
            nr_pct += '  ⚠ median unreliable'
        elif boot['nr_fraction'] > 0.10:
            nr_pct += '  ⚠ borderline'

        median_rows.append({
            'Dataset'              : label,
            'Median (days)'        : _fmt(boot['point_estimate']),
            '95% CI Lower (boot)'  : _fmt(boot['ci_lower']),
            '95% CI Upper (boot)'  : _fmt(boot['ci_upper']),
            'NR fraction'          : nr_pct,
            'Estimable resamples'  : f"{boot['n_estimable']}/{n_bootstrap}",
        })

    median_df = pd.DataFrame(median_rows).set_index('Dataset')
    risk_df   = pd.DataFrame(risk_rows, index=timepoints)
    risk_df.index.name = 'Days'

    return median_df, risk_df


In [15]:
datasets = {
    'Real':  prepare_survival(df_real,  time_col, event_col)[0],
    'GC':    prepare_survival(df_gc,    time_col, event_col)[0],
    'TVAE':  prepare_survival(df_vae,   time_col, event_col)[0],
    'CTGAN': prepare_survival(df_gan,   time_col, event_col)[0],
    'C-GAN': prepare_survival(df_gcgan, time_col, event_col)[0],
}

median_df, risk_df = compute_km_statistics(datasets, n_bootstrap=2000)

print('=== Median Survival with Bootstrap 95% CI ===')
print(median_df.to_string())
print()
print('NR fraction interpretation')
print('  < 10%  : median stable, CI reliable')
print('  10-40% : borderline — CI computed on subset of resamples (optimistic)')
print('  > 40%  : median unreliable — consider RMST or survival at fixed time')
print()
print('=== Numbers at Risk ===')
print(risk_df.to_string())

median_df.to_csv('km_median_bootstrap_ci.csv')
risk_df.to_csv('km_numbers_at_risk.csv')


Bootstrap KM:   0%|          | 0/5 [00:00<?, ?it/s]

=== Median Survival with Bootstrap 95% CI ===
         Median (days)  95% CI Lower (boot)  95% CI Upper (boot)          NR fraction Estimable resamples
Dataset                                                                                                  
Real            2839.0               2261.0               3013.0  13.7%  ⚠ borderline           1727/2000
GC              2988.0               2688.0               3264.0                 0.0%           2000/2000
TVAE            2613.0               2387.0               2840.0                 0.1%           1999/2000
CTGAN           2746.0               2613.0               3028.0                 0.1%           1999/2000
C-GAN           2855.0               2705.0               3061.0                 3.1%           1938/2000

NR fraction interpretation
  < 10%  : median stable, CI reliable
  10-40% : borderline — CI computed on subset of resamples (optimistic)
  > 40%  : median unreliable — consider RMST or survival at fixed time

==